In [3]:
import json, random, time
from datetime import datetime, timezone
from confluent_kafka import Producer

BOOTSTRAP = "localhost:9092,localhost:9094,localhost:9096"
TOPIC = "urbanpulse.bus_gps"
ROUTES = [f"RT-{i:03d}" for i in range(1, 41)]
LAT_RANGE, LON_RANGE = (12.85, 13.10), (77.50, 77.75)

producer = Producer({
    "bootstrap.servers": BOOTSTRAP,
    "acks": "all",
    "enable.idempotence": True,
    "retries": 5,
    "linger.ms": 5,
})

def make_event(route_id, bus_id):
    return {
        "bus_id": bus_id, "route_id": route_id,
        "lat": round(random.uniform(*LAT_RANGE), 6),
        "lon": round(random.uniform(*LON_RANGE), 6),
        "speed_kmh": round(random.uniform(0, 60), 1),
        "occupancy_pct": round(random.uniform(10, 100), 1),
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }

def run(duration_sec=30, target_rate=200):
    bus_ids = {r: [f"{r}-BUS-{i:03d}" for i in range(50)] for r in ROUTES}
    end = time.time() + duration_sec
    sent = 0
    while time.time() < end:
        route_id = random.choice(ROUTES)
        bus_id = random.choice(bus_ids[route_id])
        event = make_event(route_id, bus_id)
        producer.produce(TOPIC, key=route_id.encode(), value=json.dumps(event).encode())
        producer.poll(0)
        sent += 1
        time.sleep(1.0 / target_rate)
    producer.flush(10)
    print(f"sent {sent} events")

run()

sent 5697 events


In [4]:
import json, random, time
from datetime import datetime, timezone
from confluent_kafka import Producer

BOOTSTRAP = "localhost:9092,localhost:9094,localhost:9096"
TOPIC = "urbanpulse.bus_gps"
ROUTES = [f"RT-{i:03d}" for i in range(1, 41)]
LAT_RANGE, LON_RANGE = (12.85, 13.10), (77.50, 77.75)

producer = Producer({
    "bootstrap.servers": BOOTSTRAP,
    "acks": "all",
    "enable.idempotence": True,
    "retries": 5,
    "linger.ms": 5,
})

def make_event(route_id, bus_id):
    bad_gps = random.random() < 0.04   # ~4% simulated GPS glitch
    lat = round(random.uniform(12.5, 13.4), 6) if bad_gps else round(random.uniform(*LAT_RANGE), 6)
    lon = round(random.uniform(77.2, 78.0), 6) if bad_gps else round(random.uniform(*LON_RANGE), 6)
    return {
        "bus_id": bus_id, "route_id": route_id,
        "lat": lat, "lon": lon,
        "speed_kmh": round(random.uniform(0, 60), 1),
        "occupancy_pct": round(random.uniform(10, 100), 1),
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }

def run(duration_sec=300, target_rate=200):
    bus_ids = {r: [f"{r}-BUS-{i:03d}" for i in range(50)] for r in ROUTES}
    end = time.time() + duration_sec
    sent = 0
    while time.time() < end:
        route_id = random.choice(ROUTES)
        bus_id = random.choice(bus_ids[route_id])
        event = make_event(route_id, bus_id)
        producer.produce(TOPIC, key=route_id.encode(), value=json.dumps(event).encode())
        producer.poll(0)
        sent += 1
        time.sleep(1.0 / target_rate)
    producer.flush(10)
    print(f"sent {sent} events")

run()

sent 57009 events
